**Verileri Okuyup Analiz Etme**

In [79]:
import pandas as pd
import numpy as np
import re

# Dosya yolları (raw data klasörü - Görseldeki yapıya sadık kalıyoruz)
PATH_TESLIM = "../data/raw/dogo_teslim_edilenler.xlsx"
PATH_IADE = "../data/raw/dogo_iade.xlsx"
PATH_IPTAL = "../data/raw/dogo_iptal.xlsx"
PATH_ACIKLAMALI = "../data/raw/dogo_iade_aciklamali.xlsx"

def clean_column_names(df):
    """
    Sütun isimlerini veri boru hattı (pipeline) standartlarına uygun hale getirir:
    - Baştaki ve sondaki gereksiz boşlukları (whitespace) siler.
    - Türkçe karakterleri İngilizce karşılıklarına dönüştürür.
    - Tüm harfleri küçük harfe çevirir.
    - Boşlukları ve özel karakterleri alt çizgi (_) ile değiştirir.
    """
    turkish_map = str.maketrans("ğüşöçıİĞÜŞÖÇ", "gusoiciGUSOC")
    
    new_columns = []
    for col in df.columns:
        col = str(col).strip() # Baş/son boşlukları al
        col = col.translate(turkish_map) # Türkçe karakterleri dönüştür
        col = col.lower() # Küçük harfe çevir
        col = re.sub(r'[^a-z0-9]+', '_', col) # Alfanümerik olmayanları alt çizgi yap
        col = col.strip('_') # Başta/sonda alt çizgi kaldıysa temizle
        new_columns.append(col)
        
    df.columns = new_columns
    return df

# Verileri okuma 
print("Excel dosyaları belleğe alınıyor, lütfen bekleyin...")
df_teslim = pd.read_excel(PATH_TESLIM)
df_iade = pd.read_excel(PATH_IADE)
df_iptal = pd.read_excel(PATH_IPTAL)
df_aciklamali = pd.read_excel(PATH_ACIKLAMALI)

# Sütun isimlerini standartlaştırma işlemini uygulama
df_teslim = clean_column_names(df_teslim)
df_iade = clean_column_names(df_iade)
df_iptal = clean_column_names(df_iptal)
df_aciklamali = clean_column_names(df_aciklamali)

print("-" * 50)
print("Tüm veriler başarıyla okundu ve sütun isimleri normalize edildi!")
print(f"Teslim Edilenler Veri Boyutu: {df_teslim.shape}")
print(f"İade Veri Boyutu: {df_iade.shape}")
print(f"İptal Veri Boyutu: {df_iptal.shape}")
print(f"Açıklamalı İade Veri Boyutu: {df_aciklamali.shape}")

Excel dosyaları belleğe alınıyor, lütfen bekleyin...
--------------------------------------------------
Tüm veriler başarıyla okundu ve sütun isimleri normalize edildi!
Teslim Edilenler Veri Boyutu: (2670, 62)
İade Veri Boyutu: (234, 62)
İptal Veri Boyutu: (270, 62)
Açıklamalı İade Veri Boyutu: (498, 17)


**Tekilleştirme ve Ana Tabloyu Oluşturma**

In [80]:
# Her veriye kendi orijinal durumunu belirten bir etiket (flag) ekliyoruz
df_teslim['siparis_durumu_orijinal'] = 'Teslim Edildi'
df_iade['siparis_durumu_orijinal'] = 'İade Edildi'
df_iptal['siparis_durumu_orijinal'] = 'İptal Edildi'

# İlk 3 tabloyu tek bir ana "Siparişler" (orders) tablosunda alt alta birleştiriyoruz
df_orders = pd.concat([df_teslim, df_iade, df_iptal], ignore_index=True)

print("Üç ana tablo başarıyla birleştirildi!")
print(f"Birleştirilmiş Ana Sipariş Tablosu Boyutu: {df_orders.shape}")

# Sipariş numaralarında tekrar eden (duplicate) kayıt var mı kontrol edelim
# İleride 'siparis_no' üzerinden eşleşme yapacağımız için bu kolonun eşsiz (unique) olması çok önemli.
duplicate_orders = df_orders[df_orders.duplicated(subset=['siparis_no'], keep=False)]

print("-" * 50)
if len(duplicate_orders) > 0:
    print(f"DİKKAT: Ana tabloda {len(duplicate_orders)} adet tekrar eden 'siparis_no' bulundu!")
    print("Mükerrer (çift) kayıtların durumları:")
    print(duplicate_orders['siparis_durumu_orijinal'].value_counts())
else:
    print("Harika! Ana tabloda tekrar eden hiçbir sipariş numarası yok. Veri temiz.")

Üç ana tablo başarıyla birleştirildi!
Birleştirilmiş Ana Sipariş Tablosu Boyutu: (3174, 63)
--------------------------------------------------
Harika! Ana tabloda tekrar eden hiçbir sipariş numarası yok. Veri temiz.


**Sessiz İadeler**

In [81]:
# Açıklamalı iade dosyasındaki eşsiz sipariş numaralarını alalım
aciklamali_siparis_nolar = df_aciklamali['siparis_no'].dropna().unique()

# Ana tablomuzda 'Teslim Edildi' görünüp, aslında açıklamalı listede olan siparişleri tespit edelim
sessiz_iadeler = df_orders[
    (df_orders['siparis_durumu_orijinal'] == 'Teslim Edildi') & 
    (df_orders['siparis_no'].isin(aciklamali_siparis_nolar))
]

print(f"Tespit Edilen 'Sessiz İade' Sipariş Sayısı: {len(sessiz_iadeler)}")

# Yeni ve tertemiz (gerçekleri yansıtan) bir durum kolonu oluşturalım
df_orders['siparis_durumu_guncel'] = df_orders['siparis_durumu_orijinal']

# Yakaladığımız sessiz iadelerin güncel durumunu 'İade Edildi' (veya Mutabakat İadesi) olarak değiştirelim
# (Not: İleride analizi ayırabilmek için buna 'Sessiz İade' adını veriyoruz)
df_orders.loc[sessiz_iadeler.index, 'siparis_durumu_guncel'] = 'Sessiz İade'

print("-" * 50)
print("MUTABAKAT SONRASI GÜNCEL SİPARİŞ DURUMLARI:")
print(df_orders['siparis_durumu_guncel'].value_counts())

Tespit Edilen 'Sessiz İade' Sipariş Sayısı: 100
--------------------------------------------------
MUTABAKAT SONRASI GÜNCEL SİPARİŞ DURUMLARI:
siparis_durumu_guncel
Teslim Edildi    2570
İptal Edildi      270
İade Edildi       234
Sessiz İade       100
Name: count, dtype: int64


**Finansal Veri ve Tarih Normalizasyonu**

In [82]:
def clean_financial_and_date_cols(df):
    """
    Finansal sütunlardaki kuruş ve binlik ayırıcı hatalarını güvenli bir şekilde çözer.
    """
    financial_keywords = ['tutar', 'kdv', 'kargo', 'fiyat', 'bedel', 'kur']
    financial_cols = [col for col in df.columns if any(keyword in col for keyword in financial_keywords)]
    
    for col in financial_cols:
        if df[col].dtype == 'object':
            # Sadece virgül varsa noktaya çevir (1234,56 -> 1234.56)
            # Eğer zaten nokta varsa (3129.97) HİÇ DOKUNMA! Noktayı silmiyoruz.
            df[col] = df[col].astype(str).str.replace(',', '.', regex=False)
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Tarih sütunlarını datetime'a çevir
    date_cols = [col for col in df.columns if 'tarih' in col]
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors='coerce')
        
    return df

# Fonksiyonu ana sipariş tablomuza uyguluyoruz
df_orders = clean_financial_and_date_cols(df_orders)

print("✅ Finansal (Para) ve Tarih (Zaman) sütunları standart veri tiplerine dönüştürüldü!")

✅ Finansal (Para) ve Tarih (Zaman) sütunları standart veri tiplerine dönüştürüldü!


**Kategorik Metin Tutarsızlıkları**

In [83]:
def clean_categorical_texts(df):
    """
    Metin tabanlı sütunlardaki baş/son boşlukları siler ve 
    hepsini BÜYÜK HARF formatına çevirerek standartlaştırır.
    """
    # Sadece metin içeren sütunları seçelim
    text_cols = df.select_dtypes(include=['object', 'string']).columns
    
    for col in text_cols:
        # Metni standartlaştır: boşlukları al ve büyük harfe çevir
        df[col] = df[col].astype(str).str.strip().str.upper()
        # 'NAN', 'NONE' gibi metne dönüşmüş boş değerleri gerçek null (NaN) değere geri çevir
        df[col] = df[col].replace({'NAN': np.nan, 'NONE': np.nan, 'NULL': np.nan})
        
    # Gözden kaçabilecek diğer sayısal sütunları zorla sayı yapalım
    if 'gecen_sure_dk' in df.columns:
        df['gecen_sure_dk'] = pd.to_numeric(df['gecen_sure_dk'], errors='coerce')
        
    return df

# Fonksiyonu uygulayalım
df_orders = clean_categorical_texts(df_orders)

print("Kategorik metinler standartlaştırıldı (Büyük Harf ve Boşluksuz)!")
print("-" * 50)

# Aşama 1 - Madde 2: Veri Kalitesi (Eksik Veri) Raporu
missing_data = df_orders.isnull().sum()
missing_data = missing_data[missing_data > 0].sort_values(ascending=False)
missing_percentage = (missing_data / len(df_orders)) * 100

missing_df = pd.DataFrame({
    'Eksik_Satir_Sayisi': missing_data,
    'Eksik_Orani_(%)': missing_percentage.round(2)
})

print("VERİ KALİTESİ RAPORU: En çok eksik veriye (NaN) sahip sütunlar:")
print(missing_df.head(15))

Kategorik metinler standartlaştırıldı (Büyük Harf ve Boşluksuz)!
--------------------------------------------------
VERİ KALİTESİ RAPORU: En çok eksik veriye (NaN) sahip sütunlar:
                     Eksik_Satir_Sayisi  Eksik_Orani_(%)
firma_uye                          3174           100.00
kargo_no                           3174           100.00
uye_temsilci                       3174           100.00
genel_siparis_notu                 3174           100.00
hopi_paracck                       3174           100.00
hopi_birdid                        3174           100.00
paracck_tutarc                     3174           100.00
hopi_kampanya                      3174           100.00
firma_fatura                       3174           100.00
platform_siparis_no                3174           100.00
vergi_dairesi                      3147            99.15
vergi_tc_no                        3145            99.09
alt_odeme_tipi                     2959            93.23
hizmet_bedeli         

**Çöp Sütunları Temizleme**

In [84]:
# Yüzde 90'dan fazlası boş olan sütunları belirleme
threshold = 90.0
cols_to_drop = missing_percentage[missing_percentage >= threshold].index.tolist()

# Bu sütunları ana sipariş tablosundan (df_orders) kalıcı olarak silme
df_orders = df_orders.drop(columns=cols_to_drop)

print(f"Toplam {len(cols_to_drop)} adet gereksiz/boş sütun silindi.")
print("Silinen Sütunlar:", cols_to_drop)
print("-" * 50)
print(f"Güncel Ana Sipariş Tablosu Boyutu: {df_orders.shape}")

Toplam 13 adet gereksiz/boş sütun silindi.
Silinen Sütunlar: ['firma_uye', 'kargo_no', 'uye_temsilci', 'genel_siparis_notu', 'hopi_paracck', 'hopi_birdid', 'paracck_tutarc', 'hopi_kampanya', 'firma_fatura', 'platform_siparis_no', 'vergi_dairesi', 'vergi_tc_no', 'alt_odeme_tipi']
--------------------------------------------------
Güncel Ana Sipariş Tablosu Boyutu: (3174, 51)


**Son Denetim**

In [85]:
# 1. Finansal boşlukları (NaN) sıfır (0) ile doldurma
# Eğer kargo_toplamc veya hizmet_bedeli boşsa, müşteri bu bedelleri ödememiştir.
finansal_sutunlar = [col for col in df_orders.columns if any(kelime in col for kelime in ['tutar', 'kdv', 'kargo', 'fiyat', 'bedel', 'kur'])]

for col in finansal_sutunlar:
    if df_orders[col].isnull().any():
        df_orders[col] = df_orders[col].fillna(0)

print("Finansal sütunlardaki boşluklar (NaN) başarıyla 0 ile dolduruldu.")
print("-" * 50)

# 2. Mantıksal Aykırı Değer (Outlier) Kontrolü
# Sayısal sütunlarda negatif değer veya mantıksız maksimum değer var mı diye bakıyoruz
sayisal_sutunlar = df_orders.select_dtypes(include=['float64', 'int64']).columns

print("AYKIRI DEĞER VE MANTIK KONTROLÜ:")
for col in sayisal_sutunlar:
    min_val = df_orders[col].min()
    max_val = df_orders[col].max()
    
    # Eğer bir sütunda minimum değer sıfırın altındaysa (negatifse) bizi uyar
    if min_val < 0:
        hatali_satir_sayisi = len(df_orders[df_orders[col] < 0])
        print(f"DİKKAT: '{col}' sütununda negatif değerler var! (Minimum: {min_val}, Hatalı Satır: {hatali_satir_sayisi})")
    
    # Sadece belli başlı süre/tutar kolonlarının minimum ve maksimumunu bilgi amaçlı yazdır
    if col in ['gecen_sure_dk', 'tutar', 'kargo_toplamc']:
        print(f"ℹ️ {col} -> Min: {min_val} | Max: {max_val}")

print("-" * 50)
print(f"Son Durumda DataFrame Boyutu: {df_orders.shape}")
print("Veri setimizde hiç 'NaN' kalan sayısal sütun var mı?:", df_orders[sayisal_sutunlar].isnull().any().any())

Finansal sütunlardaki boşluklar (NaN) başarıyla 0 ile dolduruldu.
--------------------------------------------------
AYKIRI DEĞER VE MANTIK KONTROLÜ:
ℹ️ tutar -> Min: 0.0 | Max: 17621.99
ℹ️ kargo_toplamc -> Min: 0.0 | Max: 1067.84
--------------------------------------------------
Son Durumda DataFrame Boyutu: (3174, 51)
Veri setimizde hiç 'NaN' kalan sayısal sütun var mı?: False


**Gereksiz Noise Üreten Sütunlar**

In [86]:
# Modelin işine yaramayacak Kişisel Veriler (PII) ve Benzersiz Numaralar
gereksiz_sutunlar = [
    'uye_adi', 'cep_telefonu_uye', 'e_posta_adresi', 
    'ad_teslimat', 'cep_telefonu_teslimat', 'adres_teslimat',
    'ad_fatura', 'cep_telefonu_fatura', 'adres_fatura',
    'kargo_takip_no', 'fatura_numarasi', 'irsaliye_numarasi'
]

# Listede olan ama belki daha önceki adımlarda silinmiş veya ismi farklılaşmış 
# olabilecek sütunları güvenli bir şekilde silmek için sadece var olanları seçiyoruz
silinecekler = [col for col in gereksiz_sutunlar if col in df_orders.columns]

df_orders = df_orders.drop(columns=silinecekler)

print(f"Toplam {len(silinecekler)} adet gereksiz/kişisel veri sütunu silindi.")
print("Silinen Sütunlar:", silinecekler)
print("-" * 50)
print(f"Bomba gibi, tertemiz güncel DataFrame Boyutu: {df_orders.shape}")

Toplam 9 adet gereksiz/kişisel veri sütunu silindi.
Silinen Sütunlar: ['cep_telefonu_uye', 'e_posta_adresi', 'ad_teslimat', 'cep_telefonu_teslimat', 'adres_teslimat', 'ad_fatura', 'cep_telefonu_fatura', 'adres_fatura', 'kargo_takip_no']
--------------------------------------------------
Bomba gibi, tertemiz güncel DataFrame Boyutu: (3174, 42)


In [87]:
# Kesin olarak çöpe atacağımız sütunların tam listesi
cop_sutunlar = [
    'id', 'uye_adc', 'firma_uye_adc_fatura', 'fatura_numarasc', 'irsaliye_numarasc',
    'doviz_cinsi', 'sistem_kuru', 'doviz_tutar', 'kur_fiyatc', 'kdv',
    'il_fatura', 'ilie_fatura', 'semt_fatura', 'ulke_fatura', 'posta_kodu_teslimat',
    'siparis_sureci', 'siparis_durumu_orijinal'
]

# Mevcut olanları güvenli bir şekilde silme
silinecekler_guncel = [col for col in cop_sutunlar if col in df_orders.columns]
df_orders = df_orders.drop(columns=silinecekler_guncel)

print(f"Toplam {len(silinecekler_guncel)} adet gereksiz sütun daha silindi.")
print("-" * 50)
print(f"Hedef Odaklı Güncel DataFrame Boyutu: {df_orders.shape}")

# Kalan saf ve değerli öznitelikleri (features) görelim
print("\nModelin ve Analizin Besleneceği Kalan Sütunlar:")
print(df_orders.columns.tolist())

Toplam 17 adet gereksiz sütun daha silindi.
--------------------------------------------------
Hedef Odaklı Güncel DataFrame Boyutu: (3174, 25)

Modelin ve Analizin Besleneceği Kalan Sütunlar:
['siparis_no', 'uye_grup_kodu', 'uye_grubu', 'uye_ws_kodu', 'il_teslimat', 'ilie_teslimat', 'semt_teslimat', 'ulke_teslimat', 'tutar', 'kargo_toplamc', 'hizmet_bedeli', 'kargo', 'odeme_tipi', 'banka', 'kart', 'pos', 'geien_sure_dk', 'tarih', 'platform', 'kaynak', 'aracc', 'hediye_ceki', 'fatura_tarihi', 'kampanya', 'siparis_durumu_guncel']


In [88]:
# İsimlerdeki harf kaymalarını ve anlamsızlıkları düzelten sözlük
isim_duzeltmeleri = {
    'ilie_teslimat': 'ilce_teslimat',
    'kargo_toplamc': 'kargo_toplami',
    'geien_sure_dk': 'gecen_sure_dk',
    'aracc': 'araci',
    'tarih': 'siparis_tarihi',             # Sadece tarih yerine ne tarihi olduğunu netleştirdik
    'kargo': 'kargo_firmasi',              # Daha anlaşılır
    'siparis_durumu_guncel': 'siparis_durumu' # İsim kalabalığını attık
}

# Sütun isimlerini güncelleme
df_orders = df_orders.rename(columns=isim_duzeltmeleri)

print("Kolon isimlerindeki harf kaymaları düzeltildi ve isimler pırıl pırıl oldu!")
print("-" * 50)
print("Son Şeklini Alan Sütunlarımız:")
print(df_orders.columns.tolist())

Kolon isimlerindeki harf kaymaları düzeltildi ve isimler pırıl pırıl oldu!
--------------------------------------------------
Son Şeklini Alan Sütunlarımız:
['siparis_no', 'uye_grup_kodu', 'uye_grubu', 'uye_ws_kodu', 'il_teslimat', 'ilce_teslimat', 'semt_teslimat', 'ulke_teslimat', 'tutar', 'kargo_toplami', 'hizmet_bedeli', 'kargo_firmasi', 'odeme_tipi', 'banka', 'kart', 'pos', 'gecen_sure_dk', 'siparis_tarihi', 'platform', 'kaynak', 'araci', 'hediye_ceki', 'fatura_tarihi', 'kampanya', 'siparis_durumu']


**Feature Enginnering Tarafı**

In [89]:
# 1. Teslimat Gecikme Skoru (Gün Bazında)
# Fatura tarihi ile sipariş tarihi arasındaki gün farkı
df_orders['gecikme_gun_sayisi'] = (df_orders['fatura_tarihi'] - df_orders['siparis_tarihi']).dt.days
# Eğer negatif bir gün çıkarsa (mantık hatası veya sistem hatası), onu 0 yapalım
df_orders['gecikme_gun_sayisi'] = df_orders['gecikme_gun_sayisi'].apply(lambda x: x if x >= 0 else 0)

# 2. Sepet Büyüklüğü Kategorisi
# Toplam tutarları adil bir şekilde 3 parçaya bölüyoruz (Düşük, Orta, Yüksek Segment)
df_orders['sepet_segmenti'] = pd.qcut(df_orders['tutar'], q=3, labels=['Düşük', 'Orta', 'Yüksek'])

# 3. Zamanın Etkisi (Ay, Gün ve Hafta Sonu Flag'i)
df_orders['siparis_ayi'] = df_orders['siparis_tarihi'].dt.month
df_orders['siparis_gunu'] = df_orders['siparis_tarihi'].dt.day_name()
# Cumartesi (5) ve Pazar (6) ise 1, değilse 0
df_orders['haftasonu_mu'] = df_orders['siparis_tarihi'].dt.dayofweek.isin([5, 6]).astype(int)

# 4. İndirim / Kampanya Etkisi
# 'kampanya' sütunu boş (NaN) değilse bu bir kampanya satışıdır (1), boşsa normal satıştır (0)
df_orders['kampanyali_mi'] = df_orders['kampanya'].notnull().astype(int)

print("Feature Engineering (Öznitelik Mühendisliği) başarıyla tamamlandı!")
print("-" * 50)
print("Yeni Eklenen Süper Özniteliklerden Bir Kesit:")
print(df_orders[['siparis_no', 'gecikme_gun_sayisi', 'sepet_segmenti', 'haftasonu_mu', 'kampanyali_mi']].head())

Feature Engineering (Öznitelik Mühendisliği) başarıyla tamamlandı!
--------------------------------------------------
Yeni Eklenen Süper Özniteliklerden Bir Kesit:
    siparis_no  gecikme_gun_sayisi sepet_segmenti  haftasonu_mu  kampanyali_mi
0  TS160292132                 1.0           Orta             0              1
1  TS070392996                 2.0          Düşük             1              1
2  TS190292269                 0.0         Yüksek             0              1
3  TS090393074                 1.0         Yüksek             0              1
4  TS010493686                 0.0          Düşük             0              1


In [90]:
# 1. Eğer 'gecen_sure_dk' hala ham veri olarak duruyorsa onu güne çeviriyoruz
if 'gecen_sure_dk' in df_orders.columns:
    df_orders['gecen_sure_dk'] = pd.to_numeric(df_orders['gecen_sure_dk'], errors='coerce')
    df_orders['teslimat_suresi_gun'] = (df_orders['gecen_sure_dk'] / 1440).round(1)
    
    # İşimizi bitirdiğimiz dakika sütununu artık güvenle silebiliriz
    df_orders = df_orders.drop(columns=['gecen_sure_dk'])
    print("✅ 'teslimat_suresi_gun' başarıyla oluşturuldu ve eski dakika sütunu silindi!")
else:
    print("ℹ️ Dakika sütunu zaten silinmiş, umarım 'teslimat_suresi_gun' içerde duruyordur.")

# 2. Veriyi Türkiye Excel formatında (noktalı virgül ve virgül ile) kaydediyoruz
kayit_yolu = "../data/processed/dogo_temiz_veri_final.csv"
df_orders.to_csv(kayit_yolu, index=False, sep=';', decimal=',')

print(f"Bomba gibi veri seti başarıyla kaydedildi: {kayit_yolu}")
print(f"Mevcut Sütunlar: {df_orders.columns.tolist()}")

✅ 'teslimat_suresi_gun' başarıyla oluşturuldu ve eski dakika sütunu silindi!
Bomba gibi veri seti başarıyla kaydedildi: ../data/processed/dogo_temiz_veri_final.csv
Mevcut Sütunlar: ['siparis_no', 'uye_grup_kodu', 'uye_grubu', 'uye_ws_kodu', 'il_teslimat', 'ilce_teslimat', 'semt_teslimat', 'ulke_teslimat', 'tutar', 'kargo_toplami', 'hizmet_bedeli', 'kargo_firmasi', 'odeme_tipi', 'banka', 'kart', 'pos', 'siparis_tarihi', 'platform', 'kaynak', 'araci', 'hediye_ceki', 'fatura_tarihi', 'kampanya', 'siparis_durumu', 'gecikme_gun_sayisi', 'sepet_segmenti', 'siparis_ayi', 'siparis_gunu', 'haftasonu_mu', 'kampanyali_mi', 'teslimat_suresi_gun']


In [91]:
# Artık işimize yaramayan dakika sütununu uçuruyoruz
if 'gecen_sure_dk' in df_orders.columns:
    df_orders = df_orders.drop(columns=['gecen_sure_dk'])
    print("🗑️ 'gecen_sure_dk' sütunu başarıyla silindi!")

# Fiyatları düzelttiysen, temiz verinin son halini tekrar kaydedelim
# df_orders.to_csv("../data/processed/dogo_temiz_veri_final.csv", index=False)

In [92]:
print("Güncel Sütun Listesi:")
print("-" * 50)
for i, col in enumerate(df_orders.columns, 1):
    print(f"{i}. {col}")

Güncel Sütun Listesi:
--------------------------------------------------
1. siparis_no
2. uye_grup_kodu
3. uye_grubu
4. uye_ws_kodu
5. il_teslimat
6. ilce_teslimat
7. semt_teslimat
8. ulke_teslimat
9. tutar
10. kargo_toplami
11. hizmet_bedeli
12. kargo_firmasi
13. odeme_tipi
14. banka
15. kart
16. pos
17. siparis_tarihi
18. platform
19. kaynak
20. araci
21. hediye_ceki
22. fatura_tarihi
23. kampanya
24. siparis_durumu
25. gecikme_gun_sayisi
26. sepet_segmenti
27. siparis_ayi
28. siparis_gunu
29. haftasonu_mu
30. kampanyali_mi
31. teslimat_suresi_gun
